In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("D:\My All Documents and Projects\AML Risk Monitoring System AI powered\data\creditcard.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\ss287\AppData\Local\Temp\ipykernel_1084\1583444618.py:1: SyntaxWarning: invalid escape sequence '\M'
  df = pd.read_csv("D:\My All Documents and Projects\AML Risk Monitoring System AI powered\data\creditcard.csv")


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
df['Class'].value_counts()


Class
0    284315
1       492
Name: count, dtype: int64

We Need to balance the Highly Imbalanced data using 
PREPROCESSING (CLEAN & PREPAre Data)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("Class", axis =1)
y = df["Class"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train,X_test, y_train,y_test = train_test_split(X_scaled,y,test_size=0.2 ,stratify=y, random_state=42)

Now Building Unsupervised Model to detect Anomalies without labels

In [6]:
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.002,
    random_state=42
)

iso_model.fit(X_scaled)

df['anomaly_score'] = iso_model.decision_function(X_scaled)
df['anomaly_flag'] = iso_model.predict(X_scaled)

In [7]:
pd.crosstab(df['Class'], df['anomaly_flag'])


anomaly_flag,-1,1
Class,,
0,432,283883
1,138,354


Building Supervised Fraud Model (XGBoost) to check the Label fraud 


In [8]:
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: xgboost in f:\anaconda\lib\site-packages (3.1.3)



In [9]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

model = XGBClassifier(
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, probs)
print("ROC-AUC:", auc)


ROC-AUC: 0.984747711690191


In [10]:
pred_labels = (probs > 0.5).astype(int)
print(classification_report(y_test, pred_labels))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.64      0.86      0.73        98

    accuracy                           1.00     56962
   macro avg       0.82      0.93      0.87     56962
weighted avg       1.00      1.00      1.00     56962



In [11]:
def risk_category(score):
    if score < 30:
        return "Low"
    elif score < 70:
        return "Medium"
    else:
        return "High"

df_test = X_test.copy()
df_test = pd.DataFrame(df_test)
df_test['risk_score'] = probs * 100
df_test['risk_category'] = df_test['risk_score'].apply(risk_category)

df_test.head()


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,risk_score,risk_category
0,1.388689,-0.344345,0.852722,-0.732478,-0.938200,1.006341,-0.982113,1.524446,-0.514282,0.283673,...,1.116210,-0.359234,1.168833,-0.260585,0.093529,1.322585,0.882564,-0.261273,0.004254,Low
1,-1.578646,-1.444748,-1.674523,1.673727,-0.758956,2.059461,-1.616443,-1.451404,-0.209336,2.797579,...,0.150620,-1.302362,0.070991,-0.053061,-1.887594,0.274512,-1.550939,-0.305852,0.004465,Low
2,-0.136621,-1.825960,1.403993,0.861984,2.305066,0.817115,2.150652,1.167355,-0.601935,1.705802,...,0.023243,-0.211476,-2.450269,-0.567857,0.130276,1.368602,1.544351,-0.049095,0.032561,Low
3,0.988041,1.051919,-0.009315,-0.713961,0.272638,-0.017628,-0.806846,0.167969,-0.283116,0.414235,...,-0.881115,0.531369,-0.111589,-0.544192,0.422061,-0.157622,-0.182006,-0.349271,0.002176,Low
4,-1.180778,0.617741,0.838308,-0.886087,1.245624,0.479880,-1.586304,0.690360,-0.398512,-0.573130,...,-0.452382,-0.247624,1.022791,1.571138,-0.685416,0.116154,0.316668,-0.347232,2.072161,Low


In [12]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_test, pred_labels)
recall = recall_score(y_test, pred_labels)

print("Precision:", precision)
print("Recall:", recall)


Precision: 0.6412213740458015
Recall: 0.8571428571428571


In [13]:
import joblib

joblib.dump(model, "../models/xgb_model.pkl")
joblib.dump(iso_model, "../models/isolation_forest.pkl")


['../models/isolation_forest.pkl']